# 测试

在你完成 LangGraph agent 的原型设计后, 下一步自然是添加测试。本指南涵盖了一些在编写单元测试时可以使用的实用模式。

> **注意**
>
> 本指南是 LangGraph 专属的, 涵盖围绕具有自定义结构的 graph 的场景——如果你刚刚入门, 请查看 [测试](https://docs.langchain.com/oss/langchain/test/), 它使用 LangChain 内置的 `create_agent`。

## 前置条件

首先, 确保你已经安装了 [`pytest`](https://docs.pytest.org/):

```bash
$ pip install -U pytest
```

## 快速开始

由于许多 LangGraph agent 都依赖 state, 一种实用的模式是: 在每次用到 graph 的测试之前创建它, 然后在测试内部用一个全新的 checkpointer 实例来编译它。

下面的示例展示了它如何与一个简单的线性 graph 配合工作, 该 graph 依次经过 `node1` 和 `node2`。每个 node 都会更新单个 state 键 `my_key`:

In [1]:
# 环境准备: 从 .env 加载 API keys
from dotenv import load_dotenv

load_dotenv()  # 自动向上查找 src/oss/langgraph/.env

# ---- 统一导入 ----
import pytest

from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver


def create_graph() -> StateGraph:
    class MyState(TypedDict):
        my_key: str

    graph = StateGraph(MyState)
    graph.add_node("node1", lambda state: {"my_key": "hello from node1"})
    graph.add_node("node2", lambda state: {"my_key": "hello from node2"})
    graph.add_edge(START, "node1")
    graph.add_edge("node1", "node2")
    graph.add_edge("node2", END)
    return graph


def test_basic_agent_execution() -> None:
    checkpointer = MemorySaver()
    graph = create_graph()
    compiled_graph = graph.compile(checkpointer=checkpointer)
    result = compiled_graph.invoke(
        {"my_key": "initial_value"},
        config={"configurable": {"thread_id": "1"}}
    )
    assert result["my_key"] == "hello from node2"

In [2]:
# 端到端跑通: node1 -> node2 依次执行 (等价于上面 test_basic_agent_execution 的逻辑)
end_to_end_graph = create_graph().compile(checkpointer=MemorySaver())
result = end_to_end_graph.invoke(
    {"my_key": "initial_value"},
    config={"configurable": {"thread_id": "e2e-demo"}},
)
print(result)
assert result["my_key"] == "hello from node2"

{'my_key': 'hello from node2'}


## 测试单个 node 与 edge

编译后的 LangGraph agent 会以 `graph.nodes` 的形式暴露对每个单独 node 的引用。你可以利用这一点来测试 agent 中的单个 node。注意, 这会绕过编译 graph 时传入的任何 checkpointer:

In [3]:
# 说明: 下面这段测试与上文共用同一个 create_graph (已在上文定义, 此处直接复用)。
def test_individual_node_execution() -> None:
    # Will be ignored in this example
    checkpointer = MemorySaver()
    graph = create_graph()
    compiled_graph = graph.compile(checkpointer=checkpointer)
    # Only invoke node 1
    result = compiled_graph.nodes["node1"].invoke(
        {"my_key": "initial_value"},
    )
    assert result["my_key"] == "hello from node1"

In [4]:
# 单 node 跑通: 只调用 node1
single_node_graph = create_graph().compile(checkpointer=MemorySaver())
node1_result = single_node_graph.nodes["node1"].invoke({"my_key": "initial_value"})
print(node1_result)
assert node1_result["my_key"] == "hello from node1"

{'my_key': 'hello from node1'}


## 部分执行

对于由较大 graph 构成的 agent, 你可能希望测试 agent 内部的部分执行路径, 而不是端到端地测试整个流程。在某些情况下, 把这些部分[重构为 subgraph](https://docs.langchain.com/oss/langgraph/use-subgraphs) 在语义上是合理的, 你可以像平常一样单独调用它们。

不过, 如果你不希望改动 agent graph 的整体结构, 可以使用 LangGraph 的 persistence 机制来模拟一种 state: 让 agent 在目标部分开始之前暂停, 并在目标部分结束时再次暂停。步骤如下:

1. 用一个 checkpointer 编译你的 agent (用于测试时, 内存中的 checkpointer `InMemorySaver` 就足够了)。
2. 调用你的 agent 的 [`update_state`](https://docs.langchain.com/oss/langgraph/use-time-travel) 方法, 并把 [`as_node`](https://docs.langchain.com/oss/langgraph/use-time-travel#from-a-specific-node) 参数设置为你想要开始测试的那个 node *之前* 的那个 node 的名称。
3. 用你更新 state 时所用的同一个 `thread_id` 来调用 agent, 并把 `interrupt_after` 参数设置为你想要停止的那个 node 的名称。

下面是一个只执行线性 graph 中第二个和第三个 node 的示例:

In [5]:
# 说明: 这里的 4-node graph 与上文 create_graph 不同, 单独命名为 create_partial_graph 以免覆盖。
def create_partial_graph() -> StateGraph:
    class MyState(TypedDict):
        my_key: str

    graph = StateGraph(MyState)
    graph.add_node("node1", lambda state: {"my_key": "hello from node1"})
    graph.add_node("node2", lambda state: {"my_key": "hello from node2"})
    graph.add_node("node3", lambda state: {"my_key": "hello from node3"})
    graph.add_node("node4", lambda state: {"my_key": "hello from node4"})
    graph.add_edge(START, "node1")
    graph.add_edge("node1", "node2")
    graph.add_edge("node2", "node3")
    graph.add_edge("node3", "node4")
    graph.add_edge("node4", END)
    return graph


def test_partial_execution_from_node2_to_node3() -> None:
    checkpointer = MemorySaver()
    graph = create_partial_graph()
    compiled_graph = graph.compile(checkpointer=checkpointer)
    compiled_graph.update_state(
        config={
          "configurable": {
            "thread_id": "1"
          }
        },
        # The state passed into node 2 - simulating the state at
        # the end of node 1
        values={"my_key": "initial_value"},
        # Update saved state as if it came from node 1
        # Execution will resume at node 2
        as_node="node1",
    )
    result = compiled_graph.invoke(
        # Resume execution by passing None
        None,
        config={"configurable": {"thread_id": "1"}},
        # Stop after node 3 so that node 4 doesn't run
        interrupt_after="node3",
    )
    assert result["my_key"] == "hello from node3"

In [6]:
# 部分执行跑通: 从 node2 执行到 node3
partial_graph = create_partial_graph().compile(checkpointer=MemorySaver())
partial_graph.update_state(
    config={"configurable": {"thread_id": "partial-demo"}},
    values={"my_key": "initial_value"},
    as_node="node1",
)
partial_result = partial_graph.invoke(
    None,
    config={"configurable": {"thread_id": "partial-demo"}},
    interrupt_after="node3",
)
print(partial_result)
assert partial_result["my_key"] == "hello from node3"

{'my_key': 'hello from node3'}
